<a href="https://colab.research.google.com/github/mahathi-kannan/enterprise-data-analytics-pipeline/blob/main/01-pos-etl-pipeline/cafe_sales_cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# Load your specific file
df = pd.read_csv('dirty_cafe_sales.csv')

# Print out the initial shape (Rows, Columns)
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns.\n")

# Display missing data statistics
print("--- Missing Values Per Column ---")
print(df.isna().sum())

# Look at the column types
print("\n--- Column Data Types ---")
print(df.dtypes)

Dataset contains 10000 rows and 8 columns.

--- Missing Values Per Column ---
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

--- Column Data Types ---
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


In [3]:
# Drop any completely duplicated rows
initial_rows = len(df)
df = df.drop_duplicates()
final_rows = len(df)

print(f"Removed {initial_rows - final_rows} duplicate rows.")

Removed 0 duplicate rows.


In [4]:
# Clean 'Item' column: fill blanks and fix "UNKNOWN" strings
df['Item'] = df['Item'].replace('UNKNOWN', np.nan)
df['Item'] = df['Item'].fillna('Unknown Item')
df['Item'] = df['Item'].str.strip().str.title()

# Clean categorical columns: 'Payment Method' and 'Location'
categorical_cols = ['Payment Method', 'Location']
for col in categorical_cols:
    df[col] = df[col].replace('UNKNOWN', np.nan)
    df[col] = df[col].fillna('Not Recorded')
    df[col] = df[col].str.strip()

print("Categorical strings and placeholders standardized successfully!")

Categorical strings and placeholders standardized successfully!


In [5]:
# Force Price Per Unit and Quantity to numeric types
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(1).astype(int)
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

# If 'Price Per Unit' is missing, fill it with the median price for that specific menu item
df['Price Per Unit'] = df.groupby('Item')['Price Per Unit'].transform(lambda x: x.fillna(x.median()))

# Calculate 'Total Spent' mathematically to overwrite "ERROR" strings or missing math
df['Total Spent'] = df['Quantity'] * df['Price Per Unit']

print("Overwrote 'ERROR' strings with accurate math (Quantity x Price)!")

Overwrote 'ERROR' strings with accurate math (Quantity x Price)!


In [6]:
# Convert Transaction Date column to unified datetime format
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Check if any dates failed conversion
corrupted_dates = df['Transaction Date'].isna().sum()
print(f"Found and cleaned {corrupted_dates} broken date formats.")

Found and cleaned 460 broken date formats.


In [7]:
# Display the first 10 rows of our newly polished dataset
print("--- CLEANED DATA SNEAK PEEK ---")
print(df.head(10))

# Export to a brand new file
df.to_csv('cleaned_cafe_sales.csv', index=False)
print("\nSuccess! 'cleaned_cafe_sales.csv' has been generated.")

--- CLEANED DATA SNEAK PEEK ---
  Transaction ID          Item  Quantity  Price Per Unit  Total Spent  \
0    TXN_1961373        Coffee         2             2.0          4.0   
1    TXN_4977031          Cake         4             3.0         12.0   
2    TXN_4271903        Cookie         4             1.0          4.0   
3    TXN_7034554         Salad         2             5.0         10.0   
4    TXN_3160411        Coffee         2             2.0          4.0   
5    TXN_2602893      Smoothie         5             4.0         20.0   
6    TXN_4433211  Unknown Item         3             3.0          9.0   
7    TXN_6699534      Sandwich         4             4.0         16.0   
8    TXN_4717867  Unknown Item         5             3.0         15.0   
9    TXN_2064365      Sandwich         5             4.0         20.0   

   Payment Method      Location Transaction Date  
0     Credit Card      Takeaway       2023-09-08  
1            Cash      In-store       2023-05-16  
2     Credi